In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [ ]:
df=pd.read_csv('Cleaned_csv.csv')

In [ ]:
df.columns

In [ ]:
df.drop('Unnamed: 0',axis=1,inplace=True)

In [28]:
# Target
y = df['PotentialFraud'].astype(int)

# Features (remove target column)
X = df.drop('PotentialFraud', axis=1)


In [29]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,   # keeps 62:38 ratio
    random_state=42
)

In [30]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')  # or 'mean'
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)


In [31]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [32]:
y_pred = rf.predict(X_test)


In [33]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nROC-AUC Score:")
print(roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]))

Confusion Matrix:
[[53973 13651]
 [18876 22949]]

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.80      0.77     67624
           1       0.63      0.55      0.59     41825

    accuracy                           0.70    109449
   macro avg       0.68      0.67      0.68    109449
weighted avg       0.70      0.70      0.70    109449


ROC-AUC Score:
0.7467358050410451


In [34]:
# Fraud probability
y_prob = rf.predict_proba(X_test)[:, 1]


In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [36]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression(max_iter=3000, class_weight='balanced')


In [40]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        scale_pos_weight=(len(y_train[y_train==0]) / len(y_train[y_train==1])),
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )
}


In [41]:
results = []

for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)

    results.append([name, acc, prec, rec, f1, roc])

    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))



Training Logistic Regression...
[[39261 28363]
 [12788 29037]]
              precision    recall  f1-score   support

           0       0.75      0.58      0.66     67624
           1       0.51      0.69      0.59     41825

    accuracy                           0.62    109449
   macro avg       0.63      0.64      0.62    109449
weighted avg       0.66      0.62      0.63    109449


Training Decision Tree...
[[47954 19670]
 [18969 22856]]
              precision    recall  f1-score   support

           0       0.72      0.71      0.71     67624
           1       0.54      0.55      0.54     41825

    accuracy                           0.65    109449
   macro avg       0.63      0.63      0.63    109449
weighted avg       0.65      0.65      0.65    109449


Training Random Forest...
[[54011 13613]
 [18863 22962]]
              precision    recall  f1-score   support

           0       0.74      0.80      0.77     67624
           1       0.63      0.55      0.59     41825

  

D:\Data-Science\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [11:31:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[[43057 24567]
 [14844 26981]]
              precision    recall  f1-score   support

           0       0.74      0.64      0.69     67624
           1       0.52      0.65      0.58     41825

    accuracy                           0.64    109449
   macro avg       0.63      0.64      0.63    109449
weighted avg       0.66      0.64      0.64    109449



In [42]:
results_df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC"]
)

print("\nModel Comparison:")
print(results_df.sort_values(by="F1 Score", ascending=False))



Model Comparison:
                 Model  Accuracy  Precision    Recall  F1 Score   ROC AUC
2        Random Forest  0.703277   0.627806  0.549002  0.585765  0.746716
0  Logistic Regression  0.624017   0.505871  0.694250  0.585276  0.679324
3              XGBoost  0.639914   0.523415  0.645093  0.577919  0.688389
1        Decision Tree  0.646968   0.537459  0.546467  0.541926  0.628465


In [49]:
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
# -----------------------------------
# 1. Reduce Memory Usage
# -----------------------------------
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
# -----------------------------------
# 2. Create Sample for Tuning (100k)
# Works for both numpy and pandas
# -----------------------------------
print("Creating sample for tuning...")
sample_size = 100000
if len(X_train) > sample_size:
    indices = np.random.choice(len(X_train), sample_size, replace=False)
    X_sample = X_train[indices]    
    try:
        y_sample = y_train.iloc[indices]   # if pandas Series
    except:
        y_sample = y_train[indices]       # if numpy array
    else:
    X_sample = X_train
    y_sample = y_train
# -----------------------------------
# 3. Parameter Distribution (Light)
# -----------------------------------
param_dist = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt'],
    'bootstrap': [True]
}
# -----------------------------------
# 4. Base Model
# -----------------------------------
rf_base = RandomForestClassifier(
    class_weight='balanced',
    random_state=42
)
# -----------------------------------
# 5. Cross Validation
# -----------------------------------
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
# -----------------------------------
# 6. Randomized Search (Memory Safe)
# -----------------------------------
rf_random = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=10,
    cv=cv_strategy,
    scoring='f1',
    verbose=2,
    random_state=42,
    n_jobs=1    # IMPORTANT: prevents memory crash
)
print("Starting Hyperparameter Tuning...")
rf_random.fit(X_sample, y_sample)
print("Best Parameters:", rf_random.best_params_)
print("Best CV F1 Score:", rf_random.best_score_)
# -----------------------------------
# 7. Train Final Model on Full Data
# -----------------------------------
print("Training final model on full dataset...")
best_rf_model = RandomForestClassifier(
    **rf_random.best_params_,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
best_rf_model.fit(X_train, y_train)
# -----------------------------------
# 8. Evaluate
# -----------------------------------
y_pred = best_rf_model.predict(X_test)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))
# -----------------------------------
# 9. Save Model
# -----------------------------------


Creating sample for tuning...
Starting Hyperparameter Tuning...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END bootstrap=True, max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  19.4s
[CV] END bootstrap=True, max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  21.6s
[CV] END bootstrap=True, max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  17.6s
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  31.5s
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  32.1s
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  31.9s
[CV] END bootstrap=True, max_depth=10, max_features=s

In [2]:
import joblib
file_name="fraud_model.pkl"
joblib.dump(best_rf_model,open(file_name,'wb' ))
print("Model saved as fraud_model.pkl")

NameError: name 'best_rf_model' is not defined

In [50]:
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
# ---------------------------------
# 1. Reduce Memory Usage
# ---------------------------------
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
# ---------------------------------
# 2. Sample Data for Tuning (100k)
# Works for both numpy and pandas
# ---------------------------------
print("Creating sample for XGBoost tuning...")
sample_size = 100000
if len(X_train) > sample_size:
    indices = np.random.choice(len(X_train), sample_size, replace=False)
    X_sample = X_train[indices]    
    try:
        y_sample = y_train.iloc[indices]   # if pandas
    except:
        y_sample = y_train[indices]        # if numpy
else:
    X_sample = X_train
    y_sample = y_train
# ---------------------------------
# 3. Calculate class imbalance weight
# ---------------------------------
fraud_ratio = (y_train == 0).sum() / (y_train == 1).sum()
# ---------------------------------
# 4. Parameter Distribution (Reduced)
# ---------------------------------
xgb_params = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'scale_pos_weight': [fraud_ratio]
}
# ---------------------------------
# 5. Base Model
# ---------------------------------
xgb_base = XGBClassifier(
    eval_metric='logloss',
    random_state=42,
    tree_method='hist',   # faster + low memory
    n_jobs=1
)
# ---------------------------------
# 6. Cross Validation
# ---------------------------------
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
# ---------------------------------
# 7. Randomized Search (Memory Safe)
# ---------------------------------
xgb_random = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_params,
    n_iter=10,
    cv=cv_strategy,
    scoring='f1',
    verbose=2,
    random_state=42,
    n_jobs=1
)
print("Starting XGBoost Hyperparameter Tuning...")
xgb_random.fit(X_sample, y_sample)

print("Best XGB Params:", xgb_random.best_params_)
print("Best CV F1:", xgb_random.best_score_)
# ---------------------------------
# 8. Train Final Model on Full Data
# ---------------------------------
print("Training final XGBoost on full dataset...")
best_xgb_model = XGBClassifier(
    **xgb_random.best_params_,
    eval_metric='logloss',
    random_state=42,
    tree_method='hist',
    n_jobs=-1
)
best_xgb_model.fit(X_train, y_train)
# ---------------------------------
# 9. Evaluate
# ---------------------------------
y_pred = best_xgb_model.predict(X_test)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))
# ---------------------------------
# 10. Save Model
# ---------------------------------
joblib.dump(best_xgb_model, "xgb_fraud_model.pkl")
print("Model saved as xgb_fraud_model.pkl") 

Creating sample for XGBoost tuning...
Starting XGBoost Hyperparameter Tuning...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=3, n_estimators=200, scale_pos_weight=1.6168163967938027, subsample=1.0; total time=   3.4s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=3, n_estimators=200, scale_pos_weight=1.6168163967938027, subsample=1.0; total time=   3.1s
[CV] END colsample_bytree=1.0, learning_rate=0.05, max_depth=3, n_estimators=200, scale_pos_weight=1.6168163967938027, subsample=1.0; total time=   3.1s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=5, n_estimators=100, scale_pos_weight=1.6168163967938027, subsample=0.8; total time=   2.9s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=5, n_estimators=100, scale_pos_weight=1.6168163967938027, subsample=0.8; total time=   2.9s
[CV] END colsample_bytree=1.0, learning_rate=0.1, max_depth=5, n_estimators=100, scale_pos_weigh

In [54]:
df = pd.read_csv("Cleaned_csv.csv")

target = "PotentialFraud"
X = df.drop(columns=[target])
y = df[target]

columns = X.columns.tolist()

import joblib
joblib.dump(columns, "columns.pkl")


['columns.pkl']